# AsymMirai UQ: free-tier run (free Colab or Kaggle T4)

Runs the full evaluation on an **enriched subset**: every cancer case (up to 100) plus 3 age-matched controls each, so about 400 exams. No paid GPU needed.

**Everything is resumable.** If the session disconnects, reconnect and re-run the cells from the top. Finished downloads, PNGs and inference chunks are skipped.

Before you start:
1. Runtime → Change runtime type → **T4 GPU**.
2. In Google Drive, `MyDrive/dissertation/` should contain:
   - `weights/asymmirai_full_model.pt` (trained AsymMirai weights; link in `AsymMirai/README.md`)
   - `data/tables/EMBED_OpenData_NUS_Datathon_clinical_reduced.csv`
   - `data/tables/EMBED_OpenData_NUS_Datathon_metadata_reduced.csv`
3. Add `AWS_ACCESS_KEY_ID` and `AWS_SECRET_ACCESS_KEY` as **Colab Secrets** (key icon in the left sidebar).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
DATA_ROOT = '/content/drive/MyDrive/dissertation'   # tables, weights, PNGs and results live here (persist)
REPO = f'{DATA_ROOT}/AsymMirai_Uncertainty-Quantification'
if not os.path.exists(REPO):
    !git clone https://github.com/ShamaKolhar/AsymMirai_Uncertainty-Quantification.git "{REPO}"
else:
    !git -C "{REPO}" pull -q
os.environ['ASYMMIRAI_UQ_ROOT'] = DATA_ROOT
os.chdir(REPO); sys.path.insert(0, REPO)
!pip install -q pydicom pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg python-gdcm lifelines boto3 pyyaml

**Checkpoint:** all four lines below should print `True`. If the GPU line is `False`, change the runtime type to T4 GPU.

In [ ]:
import torch
from uq.config import load_config, set_seed
cfg = load_config('configs/free_tier.yaml')
cfg['paths']['dicom_dir'] = '/content/embed_dicoms'   # temporary local disk: DICOMs are deleted after conversion
set_seed(cfg['seed'])
print('GPU available:   ', torch.cuda.is_available())
print('weights found:   ', cfg.path('weights', 'asymmirai_full_model').exists())
print('clinical table:  ', cfg.path('clinical_csv').exists())
print('metadata table:  ', cfg.path('metadata_csv').exists())

## Step 1: choose the exams (no images needed yet)
Builds the exam table from the EMBED tables, then picks all cancer cases (up to 100) plus 3 age-matched controls per case, within the same calibration/test split.

**Checkpoint:** expect about 400 exams (about 100 cases) and 1,600 images.

In [ ]:
from uq.data import build_metadata, select_subset
build_metadata.build(cfg, plan=True)
subset = select_subset.run(cfg)

## Step 2: download only those images, then convert
PNGs are stored at the model's input size (a few MB each) on Drive. DICOMs go to temporary local disk and are deleted once converted. **If this stops partway, just re-run the cell.**

**Checkpoint:** `failed` should be 0 or close to it. Exams with missing images are skipped later.

In [ ]:
from uq.data.download_embed import download_exams
from uq.data.dicom_to_png import convert_all
download_exams(cfg, workers=8)
index = convert_all(cfg, workers=2)
!du -sh "{DATA_ROOT}/data/png16" 2>/dev/null

## Step 3: sanity check (10 exams)
**Checkpoint:** `pred_std` must be clearly above 0 (predictions differ between exams). If you see a warning about constant predictions, stop: the weights or images aren't right. `seconds_per_exam` tells you how long Step 4 will take.

In [ ]:
import json
from uq import inference
sanity_dir = inference.run(cfg, limit=10, run_name='sanity')
clean = json.load(open(sanity_dir / 'run_info.json'))['conditions']['clean']
print(clean)
n = subset['exam_id'].nunique(); spe = clean.get('seconds_per_exam', 0)
print(f"Estimated time for Step 4: ~{n * spe * len(cfg['uq']['conditions']) / 60:.0f} minutes for {n} exams")

## Step 4: full run (resumable)
Results are saved every 25 exams. **If the session disconnects, reconnect, run the setup cells, then run this cell again.** It continues where it stopped and gives identical results.

In [ ]:
run_dir = inference.run(cfg)    # run name 'free_run' comes from the config, so re-running resumes

## Step 5: evaluate
Thresholds and conformal quantiles are fitted on `calib` and reported on `test`.

In [ ]:
from uq import evaluate
from IPython.display import Markdown, Image, display
ev = evaluate.run(cfg, run_dir)
display(Markdown((ev / 'report.md').read_text()))
for f in ['roc.png', 'risk_coverage_auc.png', 'uncertainty_hist.png', 'confidence_vs_uncertainty.png',
          'conformal_sets.png', 'robustness_uncertainty.png', 'subgroup_referral.png', 'reliability.png']:
    if (ev / f).exists():
        display(Image(str(ev / f), width=650))

## Notes
- **This is an enriched case-control sample.** AUC, C-index, risk–coverage and uncertainty-vs-error are valid. Calibration (ECE, Brier) and conformal set sizes reflect the enriched prevalence, not a screening population. The report says so automatically.
- **Want more cases?** Raise `subset.max_exams` in `configs/free_tier.yaml` and re-run from Step 1. Finished work is reused.
- **Kaggle instead of Colab:** upload the tables and weights as a Kaggle Dataset, set `DATA_ROOT` to `/kaggle/working`, and use *Save & Run All* so it runs in the background without disconnecting.